# Objectif du notebook 

Date de création : 13/01/2026

Estimation des temps d'arrivée des signaux sur la voie hydro des OBS ("H"). 

In [1]:
import os
import sys
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import gc 

In [2]:
project_root = os.path.abspath(
    os.path.join(os.path.dirname(os.getcwd()), "..", "..", "..")
)
root_groix_data = os.path.join(project_root, "data", "fiberscope_groix_oct_2025")
root_groix_wav = os.path.join(root_groix_data, "wav")
root_groix_metadata = os.path.join(root_groix_data, "metadata")

root_folder = os.path.join(project_root, "real_data_analysis", "fiberscope_groix")
data_folder = os.path.join(root_folder, "data")
img_folder = os.path.join(root_folder, "img")

In [3]:
sys.path.append(project_root)
from publication.publication_figure import PubFigure, color
from real_data_analysis.fiberscope_groix.src.data_processing.arrivals_utils import *

In [4]:
pfig = PubFigure(
    label_fontsize=18, legend_fontsize=10, ticks_fontsize=16, title_fontsize=20
)

# Chargement des données utiles 

In [5]:
ds_gps = xr.open_dataset(os.path.join(data_folder, "gps.nc"))

# Exemple avec un unique signal

## Chargement des informations relatives aux émissions 

In [6]:
fpath = os.path.join(data_folder, "processed_emissions.nc")
# fpath = os.path.join(data_folder, "processed_emissions_nocorr.nc")

ds_emis = xr.open_dataset(fpath)
df_emis = ds_emis.to_dataframe()

### Remarque : calcul de la position de la source 
Le dataframe df contient la position de l'antenne GPS estimée à l'instant de l'émission. En pratique, la source n'est pas colocalisée avec l'antenne et il faut théoriquement prendre en compte ce bras de levier.

Au moins dans le cas des émissions en statique on peut considérer, au regard de l'incertitude sur le positionnement de l'antenne GPS à l'instant émission (incertitude GPS standard + interpolation linéaire entre deux points GPS), que la source est colocalisée avec l'antenne GPS. 

Dans le cas dynamique la longueur filée est importante $\approx$ 15/20 m. Dans ce cas, il peut être plus difficile de négliger le bras de levier entre l'antenne GPS et la source immergée. La longueur filée est connue ainsi que l'immersion de la source (capteur de pression sur la source), on peut ainsi calculer la distance (à la surface) de l'antenne GPS à la source (dans l'axe du navire). Néanmoins, afin de transformer ce bras de levier dans le repère du navire en un offset sur la position dans le repère ENU il est nécessaire de connaitre le cap du navire dans le repère ENU. 

Pour ce faire, on peut estimer le cap du navire à partir de l'estimation du vecteur vitesse du navire à l'instant d'émission : 

$$ V_{gps}^{(ENU)} = [V_e, V_n]^T$$

$$V_e = \frac{E_{gps}(t_{n+1}) - E_{gps}(t_n)}{\Delta t}$$
et 
$$V_n = \frac{N_{gps}(t_{n+1}) - N_{gps}(t_n)}{\Delta t}$$

où $t_n$ et $t_{n+1}$ sont les instants précèdent et successif à l'instant d'émission dans la série temporelle des positions GPS. 

Le cap du navire dans le repère ENU est alors donné par : 

$$\alpha = \arctan{\frac{V_e}{V_n}}$$

La position dans le repère du navire est (hypothèse source dans l'axe du navire):

$$X_s^{(Navire)} = [0, -bdl]^T$$

où $bdl$ est la distance, selon l'axe $y_{navire}$, de l'origine du repère du navire (position de l'antenne GPS) au projeté orthogonal de la position de la source sur la surface. 

Finalement :

$$X_s^{(ENU)} = X_{GPS}^{(ENU)} + R_{\text{Nav to ENU}} X_s^{(Navire)}$$

avec 
$$ R_{\text{Nav to ENU}} = \begin{bmatrix} \cos{\alpha} & \sin{\alpha} \\ -\sin{\alpha} & \cos{\alpha} \end{bmatrix}$$

Proche des OBS la correction pourrait avoir un impacte significatif. 

### Sélection d'une partie des émissions

In [7]:
# subset_params = {
#     "Signal": "chirp",  # "chirp" or "sinus"
#     "Source": "fixed",  # "fixed" or "trailed"
#     "Nrepeat": 10,  # number of repeats
#     "Vc carte (V)": None,  # Source amplitude (V)
#     "Emission datetime": None,  # Specific day to select (datetime object) e.g datetime(2025, 10, 16)
# }

# Old naming conventions 
# subset_params = {
#     "Signal": "chirp",  # "chirp" or "sinus"
#     "Source": None,  # "fixed" or "trailed"
#     "Nrepeat": None,  # number of repeats
#     "Vc carte (V)": None,  # Source amplitude (V)
#     "Emission datetime": None,  # Specific day to select (datetime object) e.g datetime(2025, 10, 16)
# }

# New naming conventions
subset_params = {
    "signal_type": "chirp",  # "chirp" or "sinus"
    "src_pos_status": None,  # "fixed" or "trailed"
    "num_repetitions": None,  # number of repeats
    "board_voltage_v": None,  # Source amplitude (V)
    "emission_datetime": None,  # Specific day to select (datetime object) e.g datetime(2025, 10, 16)
}


df_sel = select_dataframe_subset(df_emis, subset_params)

In [8]:
df_sel

,emission_datetime,sequence_id,src_pos_point,src_pos_status,deployed_length_m,hydrophone_distance_m,signal_type,frequency_min_hz,frequency_max_hz,duration_s,...,emission_interp_u_gps,emission_interp_e_ais,emission_interp_n_ais,emission_interp_u_ais,emission_interp_ve_gps,emission_interp_vn_gps,emission_interp_vu_gps,emission_interp_ve_ais,emission_interp_vn_ais,emission_interp_vu_ais
0,2025-10-14 09:03:32.633609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,30.523987,2916.676342,-644.734298,30.511321,0.610003,-0.261415,-0.000948,0.652855,-0.333760,-0.000977
1,2025-10-14 09:03:42.633609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,30.507687,2924.192398,-648.104521,30.494637,0.835529,-0.284358,-0.002858,0.954855,-0.302628,-0.002918
2,2025-10-14 09:03:52.633609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,30.466832,2935.773432,-650.786855,30.452960,0.787204,-0.201794,-0.002183,0.954017,-0.146505,-0.002258
3,2025-10-14 09:04:02.633609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,30.464028,2943.272739,-651.034622,30.449485,0.591203,-0.110888,0.000365,0.749931,-0.024777,0.000297
4,2025-10-14 09:04:12.633609375,1,5,trailed,14,1.70,chirp,100.0,1000.0,5.0,...,30.474122,2950.772052,-651.282391,30.458906,0.591203,-0.110888,0.002169,0.749932,-0.024777,0.002101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1982,2025-10-16 14:47:32.053187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,30.720633,-811.417761,-815.906089,30.718840,-0.600931,-0.243395,0.000387,-0.463836,-0.224460,0.000405
1983,2025-10-16 14:47:34.053187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,30.720420,-812.274100,-816.317353,30.718679,-0.600931,-0.243395,0.000867,-0.498578,-0.242801,0.000878
1984,2025-10-16 14:47:36.053187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,30.720208,-813.130440,-816.728617,30.718517,-0.600931,-0.243395,0.001346,-0.533320,-0.261142,0.001351
1985,2025-10-16 14:47:38.053187500,168,1S,fixed,8,1.35,chirp,80.0,1000.0,1.0,...,30.719995,-813.986779,-817.139881,30.718355,-0.600931,-0.243395,0.001826,-0.568063,-0.279482,0.001824


In [9]:
df_sel.columns

Index(['emission_datetime', 'sequence_id', 'src_pos_point', 'src_pos_status',
       'deployed_length_m', 'hydrophone_distance_m', 'signal_type',
       'frequency_min_hz', 'frequency_max_hz', 'duration_s', 'num_repetitions',
       'repeat_period_s', 'board_voltage_v', 'amplifier_gain', 'file_name',
       'file_start_datetime', 'start_sample', 'fsc_shift_s',
       'applied_time_shift_s', 'emission_latitude_gps',
       'emission_longitude_gps', 'emission_e_gps', 'emission_n_gps',
       'emission_u_gps', 'emission_latitude_ais', 'emission_longitude_ais',
       'emission_e_ais', 'emission_n_ais', 'emission_u_ais',
       'emission_interp_e_gps', 'emission_interp_n_gps',
       'emission_interp_u_gps', 'emission_interp_e_ais',
       'emission_interp_n_ais', 'emission_interp_u_ais',
       'emission_interp_ve_gps', 'emission_interp_vn_gps',
       'emission_interp_vu_gps', 'emission_interp_ve_ais',
       'emission_interp_vn_ais', 'emission_interp_vu_ais'],
      dtype='object')

In [10]:
df_sel.applied_time_shift_s

0       27.97
1       27.97
2       27.97
3       27.97
4       27.97
        ...  
1982    27.97
1983    27.97
1984    27.97
1985    27.97
1986    27.97
Name: applied_time_shift_s, Length: 1987, dtype: float64

In [11]:
df_sel.fsc_shift_s

0        27.898127
1        27.898127
2        27.898127
3        27.898127
4        27.898127
           ...    
1982   -522.894970
1983   -522.894970
1984   -522.894970
1985   -522.894970
1986   -522.894970
Name: fsc_shift_s, Length: 1987, dtype: float64

In [12]:
plot = False
savefig = False
verbose = False
plot_zoom = False

# Define image folder for preprocessing plots
img_process_folder = os.path.join(
    img_folder, "reception", "arrivals_detection", "processed_emissions"
)

# Convention Gen_Axes_D_V4 (Cf ELOBSBin2Wav.py)
channels_order = {
    "Z": 0,
    "X": 1,
    "Y": 2,
    "H": 3,
}
used_channel = "H"

# TODO : check to remove this or to moove it earlier
# t_hydro_source_offset = 27  # seconds

# Window parameters
pre_reception_time = 5.0  # seconds before reception to include in the window
post_reception_time = 5.0  # seconds after reception to include in the window

# # Correct window for hydrophone to source offset
# pre_reception_time -= t_hydro_source_offset
# post_reception_time += t_hydro_source_offset

sel_sequence_id = df_sel["sequence_id"].unique()

# print(sel_sequence_id)
# sel_sequence_id = [151, 116, 127, 144, 145, 146, 147]
sel_sequence_id = [146, 144, 147]

# sel_sequence_id = [str(seq_id) for seq_id in sel_sequence_id]
print(sel_sequence_id)

[146, 144, 147]


In [13]:
sel_sequence_id = df_sel["sequence_id"].unique()

# Process in batches if plot is needed to avoid memory issues (restart kernel between batches)
process_batch = False
if process_batch:
    i = 0
    batch_size = 5
    sel_sequence_id = sel_sequence_id[i * batch_size : (i + 1) * batch_size]
    plot = True
    savefig = True

df_processed = build_arrivals_dataset(
    df=df_sel,
    ds_gps=ds_gps,
    root_merged_wav=data_folder,
    sel_sequence_id=sel_sequence_id,
    pre_reception_time=pre_reception_time,
    post_reception_time=post_reception_time,
    img_root=img_process_folder,
    channels_order=channels_order,
    used_channel=used_channel,
    plot=plot,
    plot_zoom=False,
    savefig=savefig,
    verbose=False,
)

if not process_batch:
    ds_processed = xr.Dataset.from_dataframe(df_processed)
    fpath_save = os.path.join(
        data_folder,
        f"processed_arrivals.nc",
        # f"processed_arrivals_nocorr.nc",
    )
    ds_processed.to_netcdf(fpath_save)

Selected sequences: 84 -> [  1   2  12  13  14  15  16  24  25  26  27  28  29  35  39  40  41  42
  43  44  46  47  51  55  59  60  61  62  63  64  69  70  71  72  76  80
  81  82  83  84  85  90  91  95  99 100 101 102 103 104 106 110 116 117
 118 119 120 121 123 127 131 132 133 134 135 136 138 139 143 144 145 146
 147 151 155 156 157 158 159 160 162 163 167 168]
Progress: █................................................................................................... 1%
Progress: ██.................................................................................................. 2%
Progress: ███████............................................................................................. 7%
Progress: ████████............................................................................................ 8%


Progress: █████████████....................................................................................... 13%

Progress: ██████████████..................................

PermissionError: [Errno 13] Permission denied: 'c:\\Users\\baptiste.menetrier\\Desktop\\devPy\\phd\\real_data_analysis\\fiberscope_groix\\data\\processed_arrivals.nc'

In [ ]:
sel_sequence_id = [147, 146, 144]

# Process in batches if plot is needed to avoid memory issues (restart kernel between batches)
process_batch = False
if process_batch:
    i = 0 
    batch_size = 5
    sel_sequence_id = sel_sequence_id[i*batch_size:(i+1)*batch_size]
    plot = True
    savefig = True

df_processed = build_arrivals_dataset(
    df=df_sel,
    ds_gps=ds_gps,
    root_merged_wav=data_folder,
    sel_sequence_id=sel_sequence_id,
    pre_reception_time=pre_reception_time,
    post_reception_time=post_reception_time,
    img_root=img_process_folder,
    channels_order=channels_order,
    used_channel=used_channel,
    plot=plot,
    plot_zoom=False,
    savefig=savefig,
    verbose=False,
)

if not process_batch:
    ds_processed = xr.Dataset.from_dataframe(df_processed)
    fpath_save = os.path.join(
        data_folder,
        f"processed_arrivals.nc",
    )
    ds_processed.to_netcdf(fpath_save)

In [ ]:
df_processed

In [ ]:
# for sel_id in df_processed["Sequence_id"].unique():
#     df_seq = df_processed[df_processed["Sequence_id"] == sel_id]

#     for obs_id in [1, 2, 3]:

#         # First criterion: ratio of detected arrivals
#         col_name = f"Valid detection OBS{obs_id}"
#         n_detected = df_seq[col_name].sum()
#         crit_1 = n_detected / df_seq.shape[0]

#         # Second criterion: error relative to expected repetition period
#         col_name = f"Arrival datetime OBS{obs_id}"
#         t_diff_mean = df_seq[col_name].diff().mean().total_seconds()
#         repeat_period_em = df_seq["Trepeat (s)"].iloc[0]
#         crit_2 = 1 - abs(t_diff_mean - repeat_period_em) / repeat_period_em
#         if t_diff_mean < 0 or crit_2 < 0 or np.isnan(crit_2):
#             crit_2 = 0

#         # Third criterion: normalized psnr
#         col_name = f"PSNR OBS{obs_id}"
#         psnr_mean = df_seq[col_name].mean()
#         crit_3 = psnr_mean / df_processed[col_name].max()
#         if np.isnan(crit_3):
#             crit_3 = 0

#         # Final score as sum of criteria
#         final_score = (crit_1 + crit_2 + crit_3) / 3
#         print(
#             f"Sequence ID {sel_id} - OBS{obs_id} : Score = {final_score:.2f} (C1={crit_1:.2f}, C2={crit_2:.2f}, C3={crit_3:.2f})"
#         )

#         # Store final score in dataframe
#         df_processed.loc[
#             (df_processed["Sequence_id"] == sel_id),
#             f"f_score OBS{obs_id}",
#         ] = final_score




In [ ]:
# df_processed_valid = df_processed.loc[
#     df_processed["Valid detection OBS1"]
#     & df_processed["Valid detection OBS2"]
#     & df_processed["Valid detection OBS3"]
# ]